## Experiment 2: Different Ranks

In [11]:
from pathlib import Path
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openpyxl

from scipy.signal import hilbert
from tensorly.decomposition import parafac, tucker, tensor_train
from tensorly.cp_tensor import cp_to_tensor
from tensorly.tucker_tensor import tucker_to_tensor
from tensorly.tt_tensor import tt_to_tensor

In [3]:
# Candidate ranks
CP_RANKS = [1, 2, 3, 5, 8, 10, 15, 20, 30]
TUCKER_RANKS = [
    (5, 2, 10),
    (10, 3, 20),
    (15, 4, 40),
    (20, 8, 60),
    (25, 10, 80),
    (30, 12, 100),
    (30, 15, 150),
    (30, 20, 200),
]
TT_RANKS = [
    (1, 2, 2, 1),
    (1, 2, 3, 1),
    (1, 3, 3, 1),
    (1, 4, 5, 1),
    (1, 10, 10, 1),
    (1, 10, 15, 1),
    (1, 15, 20, 1),
    (1, 20, 20, 1),
]

In [16]:
# ============================= PATH ===============================
PROJECT_ROOT = Path.cwd().parent.parent

LOADED_ROOT = Path(PROJECT_ROOT/"data/preprocessed_rest_epochs")
SAVE_ROOT = Path(PROJECT_ROOT/"results/decomposition/decomposition_experiment4")

SAVE_ROOT.mkdir(parents=True, exist_ok=True)

PLOT_ROOT = SAVE_ROOT / "plots"
PLOT_ROOT.mkdir(parents=True, exist_ok=True)

files = sorted(LOADED_ROOT.rglob("*.npz"))

print(f"Found {len(files)} preprocessed files.")

if len(files) == 0:
    raise FileNotFoundError(
        f"No .npz files found in {LOADED_ROOT.resolve()}"
    )

# ============================= SETTINGS ===============================

# Candidate ranks
CP_RANKS = [1, 2, 3, 5, 8, 10, 15, 20, 30]
TUCKER_RANKS = [
    (5, 2, 10),
    (10, 3, 20),
    (15, 4, 40),
    (20, 8, 60),
    (25, 10, 80),
    (30, 12, 100),
    (30, 15, 150),
    (30, 20, 200),
]
TT_RANKS = [
    (1, 2, 2, 1),
    (1, 2, 3, 1),
    (1, 3, 3, 1),
    (1, 4, 5, 1),
    (1, 10, 10, 1),
    (1, 10, 15, 1),
    (1, 15, 20, 1),
    (1, 20, 20, 1),
]

N_ITER_MAX = 100

# ============================= FUNCTIONS ===============================

def infer_subject_task_run(file):
    """
    Expected structure:

        ../data/preprocessed/
            002/
                auditory/
                    run01_epochs.npz

    Returns:
        subject = "002"
        task    = "auditory"
        run     = "run01"
    """

    subject = file.parent.parent.name
    task = file.parent.name

    # run01_epochs.npz -> run01
    run = file.stem.replace("_epochs", "")

    return subject, task, run

def relative_reconstruction_error(X, X_hat):
    """
    Relative Frobenius reconstruction error:

        ||X - X_hat||_F / ||X||_F
    """
    numerator = np.linalg.norm(X - X_hat)
    denominator = np.linalg.norm(X)

    if denominator == 0:
        return np.nan

    return numerator / denominator

def compression_ratio(original_shape, compressed_parameters):
    """
    Fraction of parameters saved.

    Returns:
        compression_ratio = 1 - compressed/original

    Example:
        0.90 = 90% fewer parameters
    """
    original_parameters = int(np.prod(original_shape))

    if original_parameters == 0:
        return np.nan

    return 1.0 - (compressed_parameters / original_parameters)

def compression_factor(original_shape, compressed_parameters):
    """
    Original parameters / compressed parameters.

    Example:
        10 means the representation is approximately 10x smaller.
    """
    original_parameters = int(np.prod(original_shape))

    if compressed_parameters == 0:
        return np.nan

    return original_parameters / compressed_parameters

def explained_variance(X, X_hat):
    """
    Explained variance based on reconstruction error.
    """

    total_energy = np.sum(X ** 2)
    residual_energy = np.sum((X - X_hat) ** 2)

    if total_energy == 0:
        return np.nan

    return 1.0 - (residual_energy / total_energy)

def cp_parameter_count(weights, factors):
    """
    Number of parameters stored by CP decomposition.

    CP:
        weights: R
        factor matrices:
            I x R
            J x R
            K x R
    """
    count = len(weights)

    for factor in factors:
        count += factor.size

    return int(count)

def tucker_parameter_count(core, factors):
    """
    Number of parameters stored by Tucker decomposition.

    Includes:
        core tensor
        all factor matrices
    """
    count = core.size

    for factor in factors:
        count += factor.size

    return int(count)

def tt_parameter_count(tt_cores):
    """
    Number of parameters stored by TT decomposition.
    """
    return int(sum(core.size for core in tt_cores))

def calculate_snr(X):
    """
    Approximate trial-based SNR.

    The mean across trials is treated as the reproducible
    signal component.

        signal = mean across trials
        noise  = individual trial - mean

        SNR = RMS(signal) / RMS(noise)

    Returns:
        SNR in dB.
    """
    if X.shape[0] < 2:
        return np.nan

    # Average response across trials
    signal = np.mean(X, axis=0)

    # Trial-to-trial residual
    noise = X - signal

    signal_rms = np.sqrt(np.mean(signal ** 2))
    noise_rms = np.sqrt(np.mean(noise ** 2))

    if noise_rms == 0:
        return np.inf

    snr_linear = signal_rms / noise_rms

    return 20 * np.log10(snr_linear)

# ============================= STORAGE ===============================

decomposition_results = []
signal_results = []

# ============================= MAIN ===============================

for file_index, file in enumerate(files, start=1):

    print("\n" + "=" * 70)
    print(f"[{file_index}/{len(files)}] {file}")
    print("=" * 70)

    # ---------- LOAD ----------

    tensor = np.load(file, allow_pickle=True)

    X = tensor["epochs"].astype(np.float64)

    times = tensor["times"] if "times" in tensor else None
    fs = float(tensor["fs"]) if "fs" in tensor else np.nan

    print(f"Tensor shape: {X.shape}")
    print(f"Sampling frequency: {fs}")

    subject, task, run = infer_subject_task_run(file)

    print(f"Subject: {subject}")
    print(f"Task: {task}")
    print(f"Run: {run}")

    # ---------- SIGNAL METRICS ----------

    print("\nCalculating signal metrics...")

    snr = calculate_snr(X)

    print(f"SNR:  {snr:.4f} dB")

    signal_results.append({
        "File": str(file),
        "Subject": subject,
        "Task": task,
        "Run": run,
        "Trials": X.shape[0],
        "Channels": X.shape[1],
        "Timepoints": X.shape[2],
        "Sampling_Frequency_Hz": fs,
        "SNR_dB": snr,
    })

    # ---------- CP DECOMPOSITION ----------

    print("\n--- CP decomposition ---")

    cp_errors = []

    for rank in CP_RANKS:

        print(f"CP rank = {rank}")

        start_time = time.perf_counter()

        cp = parafac(
            X,
            rank=rank,
            init="random", # svd was too large
            n_iter_max=N_ITER_MAX,
            tol=1e-6,
            verbose=False
        )

        elapsed = time.perf_counter() - start_time

        weights, factors = cp

        # Reconstruct tensor
        X_hat = cp_to_tensor(cp)

        error = relative_reconstruction_error(X, X_hat)
        ev = explained_variance(X, X_hat)

        # Number of stored parameters
        parameters = cp_parameter_count(
            weights,
            factors
        )

        comp_ratio = compression_ratio(
            X.shape,
            parameters
        )

        comp_factor = compression_factor(
            X.shape,
            parameters
        )

        cp_errors.append({
            "Rank": rank,
            "Reconstruction_Error": error,
            "Compression_Ratio": comp_ratio,
        })

        decomposition_results.append({
            "File": str(file),
            "Subject": subject,
            "Task": task,
            "Run": run,
            "Method": "CP",
            "Rank": str(rank),
            "Original_Parameters": int(np.prod(X.shape)),
            "Compressed_Parameters": parameters,
            "Compression_Ratio": comp_ratio,
            "Compression_Factor": comp_factor,
            "Reconstruction_Error": error,
            "Reconstruction_Error_Percent": error * 100,
            "Explained_Variance": ev,
            "Explained_Variance_Percent": ev * 100,
            "Runtime_Seconds": elapsed,
            "SNR_dB": snr,
        })

        del X_hat

    # ---------- TUCKER DECOMPOSITION ----------

    print("\n--- Tucker decomposition ---")

    tucker_errors = []

    for rank in TUCKER_RANKS:

        actual_rank = [
            min(rank[0], X.shape[0]),
            min(rank[1], X.shape[1]),
            min(rank[2], X.shape[2]),
        ]

        print(f"Tucker rank = {actual_rank}")

        start_time = time.perf_counter()

        tucker_model = tucker(
            X,
            rank=actual_rank,
            init="random", # svd was too large
            n_iter_max=N_ITER_MAX,
            tol=1e-6
        )

        elapsed = time.perf_counter() - start_time

        core, factors = tucker_model

        X_hat = tucker_to_tensor(tucker_model)

        ev = explained_variance(X, X_hat)

        error = relative_reconstruction_error(
            X,
            X_hat
        )

        parameters = tucker_parameter_count(
            core,
            factors
        )

        comp_ratio = compression_ratio(
            X.shape,
            parameters
        )

        comp_factor = compression_factor(
            X.shape,
            parameters
        )

        tucker_errors.append({
            "Rank": rank,
            "Reconstruction_Error": error,
            "Compression_Ratio": comp_ratio,
        })

        decomposition_results.append({
            "File": str(file),
            "Subject": subject,
            "Task": task,
            "Run": run,
            "Method": "Tucker",
            "Rank": str(tuple(actual_rank)),
            "Original_Parameters": int(np.prod(X.shape)),
            "Compressed_Parameters": parameters,
            "Compression_Ratio": comp_ratio,
            "Compression_Factor": comp_factor,
            "Reconstruction_Error": error,
            "Reconstruction_Error_Percent": error * 100,
            "Explained_Variance": ev,
            "Explained_Variance_Percent": ev * 100,
            "Runtime_Seconds": elapsed,
            "SNR_dB": snr,
        })

        del X_hat

    # ---------- TT DECOMPOSITION ----------

    print("\n--- TT decomposition ---")

    tt_errors = []

    for rank in TT_RANKS:

        actual_rank = list(rank)

        print(f"TT rank = {actual_rank}")

        start_time = time.perf_counter()

        tt_model = tensor_train(
            X,
            rank=actual_rank
        )

        elapsed = time.perf_counter() - start_time

        X_hat = tt_to_tensor(tt_model)

        ev = explained_variance(X, X_hat)

        error = relative_reconstruction_error(
            X,
            X_hat
        )

        parameters = tt_parameter_count(
            tt_model.factors
        )

        comp_ratio = compression_ratio(
            X.shape,
            parameters
        )

        comp_factor = compression_factor(
            X.shape,
            parameters
        )

        tt_errors.append({
            "Rank": rank,
            "Reconstruction_Error": error,
            "Compression_Ratio": comp_ratio,
        })

        decomposition_results.append({
            "File": str(file),
            "Subject": subject,
            "Task": task,
            "Run": run,
            "Method": "TT",
            "Rank": str(tuple(actual_rank)),
            "Original_Parameters": int(np.prod(X.shape)),
            "Compressed_Parameters": parameters,
            "Compression_Ratio": comp_ratio,
            "Compression_Factor": comp_factor,
            "Reconstruction_Error": error,
            "Reconstruction_Error_Percent": error * 100,
            "Explained_Variance": ev,
            "Explained_Variance_Percent": ev * 100,
            "Runtime_Seconds": elapsed,
            "SNR_dB": snr,
        })

        del X_hat

    # ---------- SAVE ----------

    safe_subject = str(subject).replace("/", "_")
    safe_task = str(task).replace("/", "_")
    safe_run = str(run).replace("/", "_")

    prefix = (
        f"{safe_subject}_{safe_task}_{safe_run}"
    )


# ---------- DATAFRAME ----------

df_decomp = pd.DataFrame(
    decomposition_results
)

df_signal = pd.DataFrame(
    signal_results
)


# ------------ SUMMARY -------------

summary = (
    df_decomp
    .groupby(
        [
            "Subject",
            "Task",
            "Method",
            "Rank"
        ],
        dropna=False
    )
    .agg(
        Mean_Reconstruction_Error=(
            "Reconstruction_Error",
            "mean"
        ),
        Std_Reconstruction_Error=(
            "Reconstruction_Error",
            "std"
        ),
        Mean_Compressed_Parameters=(
            "Compressed_Parameters",
            "mean"
        ),
        Std_Compressed_Parameters=(
            "Compressed_Parameters",
            "std"
        ),
        Mean_Compression_Ratio=(
            "Compression_Ratio",
            "mean"
        ),
        Std_Compression_Ratio=(
            "Compression_Ratio",
            "std"
        ),
        Mean_Compression_Factor=(
            "Compression_Factor",
            "mean"
        ),
        Mean_Runtime_Seconds=(
            "Runtime_Seconds",
            "mean"
        ),
        N=("File", "count")
    )
    .reset_index()
)

# --------- TASK RECONSTRUCTION ERROR SUMMARY ---------

task_reconstruction_summary = (
    df_decomp
    .groupby(
        ["Task", "Method"],
        dropna=False
    )
    .agg(
        Mean_Reconstruction_Error=(
            "Reconstruction_Error",
            "mean"
        ),
        Std_Reconstruction_Error=(
            "Reconstruction_Error",
            "std"
        ),
        Mean_Reconstruction_Error_Percent=(
            "Reconstruction_Error_Percent",
            "mean"
        ),
        Std_Reconstruction_Error_Percent=(
            "Reconstruction_Error_Percent",
            "std"
        ),
        Mean_Compressed_Parameters=(
            "Compressed_Parameters",
            "mean"
        ),
        Mean_Compression_Ratio=(
            "Compression_Ratio",
            "mean"
        ),
        Mean_Compression_Factor=(
            "Compression_Factor",
            "mean"
        ),
        N=("File", "count")
    )
    .reset_index()
)

# --------- OVERALL SUMMARY ---------

task_epochs = (
    df_signal
    .groupby("Task")["Trials"]
    .sum()
    .to_dict()
)

for task, n_epochs in task_epochs.items():
    print(f"{task}: {n_epochs:,}")

task_reconstruction_summary["Total_Epochs"] = (
    task_reconstruction_summary["Task"]
    .map(task_epochs)
)

overall_summary = (
    df_decomp
    .groupby(
        [
            "Method",
            "Rank"
        ],
        dropna=False
    )
    .agg(
        Mean_Reconstruction_Error=(
            "Reconstruction_Error",
            "mean"
        ),
        Std_Reconstruction_Error=(
            "Reconstruction_Error",
            "std"
        ),
        Mean_Compressed_Parameters=(
            "Compressed_Parameters",
            "mean"
        ),
        Std_Compressed_Parameters=(
            "Compressed_Parameters",
            "std"
        ),
        Mean_Compression_Ratio=(
            "Compression_Ratio",
            "mean"
        ),
        Std_Compression_Ratio=(
            "Compression_Ratio",
            "std"
        ),
        Mean_Compression_Factor=(
            "Compression_Factor",
            "mean"
        ),
        Mean_Runtime_Seconds=(
            "Runtime_Seconds",
            "mean"
        ),
        N=("File", "count")
    )
    .reset_index()
)

# --------- OVERALL RANK SUMMARY ---------

overall_rank_summary = (
    df_decomp
    .groupby(
        ["Method", "Rank"],
        dropna=False
    )
    .agg(
        Mean_Reconstruction_Error=(
            "Reconstruction_Error",
            "mean"
        ),
        Std_Reconstruction_Error=(
            "Reconstruction_Error",
            "std"
        ),
        Mean_Reconstruction_Error_Percent=(
            "Reconstruction_Error_Percent",
            "mean"
        ),
        Std_Reconstruction_Error_Percent=(
            "Reconstruction_Error_Percent",
            "std"
        ),
        Mean_Compressed_Parameters=(
            "Compressed_Parameters",
            "mean"
        ),
        Std_Compressed_Parameters=(
            "Compressed_Parameters",
            "std"
        ),
        Mean_Compression_Ratio=(
            "Compression_Ratio",
            "mean"
        ),
        Std_Compression_Ratio=(
            "Compression_Ratio",
            "std"
        ),
        Mean_Compression_Factor=(
            "Compression_Factor",
            "mean"
        ),
        Mean_Runtime_Seconds=(
            "Runtime_Seconds",
            "mean"
        ),
        N=("File", "count")
    )
    .reset_index()
)

# ---------- SIGNAL METRIC SUMMARY ----------

signal_summary = (
    df_signal
    .groupby(
        ["Subject", "Task"],
        dropna=False
    )
    .agg(
        Total_Epochs=("Trials", "sum"),
        Mean_SNR_dB=("SNR_dB", "mean"),
        Std_SNR_dB=("SNR_dB", "std"),
        N=("File", "count")
    )
    .reset_index()
)

# ---------- SAVE EXCEL ----------

excel_path = SAVE_ROOT / "tensor_decomposition_results.xlsx"

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    df_decomp.to_excel(
        writer,
        sheet_name="Decomposition_Metrics",
        index=False
    )

    df_signal.to_excel(
        writer,
        sheet_name="Signal_Metrics",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Subject_Task_Summary",
        index=False
    )

    overall_summary.to_excel(
        writer,
        sheet_name="Overall_Summary",
        index=False
    )

    signal_summary.to_excel(
        writer,
        sheet_name="Signal_Summary",
        index=False
    )

    task_reconstruction_summary.to_excel(
        writer,
        sheet_name="Task_Reconstruction",
        index=False
    )

    overall_rank_summary.to_excel(
        writer,
        sheet_name="Overall_Rank_Summary",
        index=False
    )

Found 32 preprocessed files.

[1/32] /home/master/MasterThesis/OPM-MEG MPS/data/preprocessed_rest_epochs/002/auditory/run01_epochs.npz
Tensor shape: (200, 30, 1401)
Sampling frequency: 2000.0
Subject: 002
Task: auditory
Run: run01

Calculating signal metrics...
SNR:  -20.4993 dB

--- CP decomposition ---
CP rank = 1
CP rank = 2
CP rank = 3
CP rank = 5
CP rank = 8
CP rank = 10
CP rank = 15
CP rank = 20
CP rank = 30

--- Tucker decomposition ---
Tucker rank = [5, 2, 10]
Tucker rank = [10, 3, 20]
Tucker rank = [15, 4, 40]
Tucker rank = [20, 8, 60]
Tucker rank = [25, 10, 80]
Tucker rank = [30, 12, 100]
Tucker rank = [30, 15, 150]
Tucker rank = [30, 20, 200]

--- TT decomposition ---
TT rank = [1, 2, 2, 1]
TT rank = [1, 2, 3, 1]
TT rank = [1, 3, 3, 1]
TT rank = [1, 4, 5, 1]
TT rank = [1, 10, 10, 1]
TT rank = [1, 10, 15, 1]
TT rank = [1, 15, 20, 1]
TT rank = [1, 20, 20, 1]

[2/32] /home/master/MasterThesis/OPM-MEG MPS/data/preprocessed_rest_epochs/002/auditory/run02_epochs.npz
Tensor shape: 

In [17]:
PLOT_ROOT = SAVE_ROOT / "trial_plots"
PLOT_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# PLOT SETTINGS
# ============================================================

methods = ["CP", "Tucker", "TT"]

task_order = [
    "auditory",
    "somatosensory",
    "motor",
    "rest"
]

task_labels = {
    "auditory": "Auditory",
    "somatosensory": "Somatosensory",
    "motor": "Motor",
    "rest": "Rest"
}

def get_method_data(method):
    """
    Return overall rank summary for one decomposition method.

    Sorted by the actual rank value.
    """
    
    subset = (
        overall_rank_summary[
            overall_rank_summary["Method"] == method
        ]
        .copy()
        .sort_values("Rank")
    )

    return subset


# ============================================================
# 1. OVERALL RECONSTRUCTION ERROR VS COMPRESSED PARAMETERS
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5),
    sharey=True
)

for ax, method in zip(axes, methods):

    subset = get_method_data(method)

    if subset.empty:
        ax.set_title(f"{method}\n(no data)")
        continue

    subset = subset.sort_values(
        "Mean_Compressed_Parameters"
    )

    ax.plot(
        subset["Mean_Compressed_Parameters"],
        subset["Mean_Reconstruction_Error_Percent"],
        marker="o",
        linewidth=2
    )

    ax.set_xscale("log")

    ax.set_title(method)

    ax.set_xlabel(
        "Mean Number of Compressed Parameters"
    )

    ax.grid(
        True,
        alpha=0.3
    )


axes[0].set_ylabel(
    "Mean Reconstruction Error (%)"
)

fig.suptitle(
    "Overall Reconstruction Error vs Model Size",
    fontsize=15
)

plt.tight_layout()

plt.savefig(
    PLOT_ROOT / "overall_reconstruction_error_vs_parameters.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 2. OVERALL RECONSTRUCTION ERROR VS COMPRESSION RATIO
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5),
    sharey=True
)

for ax, method in zip(axes, methods):

    subset = get_method_data(method)

    if subset.empty:
        ax.set_title(f"{method}\n(no data)")
        continue

    subset = subset.sort_values(
        "Mean_Compression_Ratio"
    )

    ax.plot(
        subset["Mean_Compression_Ratio"] * 100,
        subset["Mean_Reconstruction_Error_Percent"],
        marker="o",
        linewidth=2
    )

    ax.set_title(method)

    ax.set_xlabel(
        "Mean Compression Ratio (%)"
    )

    ax.grid(
        True,
        alpha=0.3
    )


axes[0].set_ylabel(
    "Mean Reconstruction Error (%)"
)

fig.suptitle(
    "Overall Reconstruction Error vs Compression",
    fontsize=15
)

plt.tight_layout()

plt.savefig(
    PLOT_ROOT / "overall_reconstruction_error_vs_compression.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ============================================================
# 3. INDIVIDUAL DECOMPOSITION PLOTS
# ============================================================

for method in methods:

    subset = get_method_data(method)

    if subset.empty:
        continue


    # --------------------------------------------------------
    # Reconstruction error vs compression ratio
    # --------------------------------------------------------

    subset_compression = subset.sort_values(
        "Mean_Compression_Ratio"
    )

    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    ax.plot(
        subset_compression["Mean_Compression_Ratio"] * 100,
        subset_compression["Mean_Reconstruction_Error_Percent"],
        marker="o",
        linewidth=2
    )

    ax.set_xlabel(
        "Mean Compression Ratio (%)"
    )

    ax.set_ylabel(
        "Mean Reconstruction Error (%)"
    )

    ax.set_title(
        f"{method} Reconstruction Error vs Compression"
    )

    ax.grid(
        True,
        alpha=0.3
    )

    plt.tight_layout()

    plt.savefig(
        PLOT_ROOT /
        f"{method}_reconstruction_error_vs_compression.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# 4. TASK RECONSTRUCTION ERROR SUMMARY
# ============================================================

task_reconstruction_summary = (
    df_decomp
    .groupby(
        ["Task", "Method"],
        dropna=False
    )
    .agg(
        Mean_Reconstruction_Error=(
            "Reconstruction_Error",
            "mean"
        ),

        Std_Reconstruction_Error=(
            "Reconstruction_Error",
            "std"
        ),

        Mean_Reconstruction_Error_Percent=(
            "Reconstruction_Error_Percent",
            "mean"
        ),

        Std_Reconstruction_Error_Percent=(
            "Reconstruction_Error_Percent",
            "std"
        ),

        Mean_Compressed_Parameters=(
            "Compressed_Parameters",
            "mean"
        ),

        Mean_Compression_Ratio=(
            "Compression_Ratio",
            "mean"
        ),

        Mean_Compression_Factor=(
            "Compression_Factor",
            "mean"
        ),

        N=(
            "File",
            "count"
        )
    )
    .reset_index()
)

task_reconstruction_summary["Total_Epochs"] = (
    task_reconstruction_summary["Task"]
    .map(task_epochs)
)


# ============================================================
# 5. RECONSTRUCTION ERROR BY TASK
# ============================================================

fig, ax = plt.subplots(
    figsize=(11, 6)
)

x = np.arange(
    len(task_order)
)

offsets = {
    "CP": -0.25,
    "Tucker": 0.0,
    "TT": 0.25
}

for method in methods:

    subset = task_reconstruction_summary[
        task_reconstruction_summary["Method"] == method
    ]

    means = []
    stds = []

    for task in task_order:

        row = subset[
            subset["Task"] == task
        ]

        if row.empty:

            means.append(np.nan)
            stds.append(np.nan)

        else:

            means.append(
                row[
                    "Mean_Reconstruction_Error_Percent"
                ].iloc[0]
            )

            stds.append(
                row[
                    "Std_Reconstruction_Error_Percent"
                ].iloc[0]
            )

    ax.errorbar(
        x + offsets[method],
        means,
        yerr=stds,
        marker="o",
        linestyle="none",
        capsize=4,
        label=method
    )


ax.set_xticks(x)

ax.set_xticklabels([
    f"{task_labels[t]}\n"
    f"({task_epochs.get(t, 0):,} epochs)"
    for t in task_order
])

ax.set_xlabel(
    "Task"
)

ax.set_ylabel(
    "Mean Reconstruction Error (%)"
)

ax.set_title(
    "Mean Reconstruction Error Across Tasks\n"
    "(Averaged Across All Ranks)"
)

ax.grid(
    True,
    axis="y",
    alpha=0.3
)

ax.legend()

plt.tight_layout()

plt.savefig(
    PLOT_ROOT /
    "reconstruction_error_by_task.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 6. SNR BY TASK
# ============================================================

task_snr = (
    df_signal
    .groupby("Task")
    .agg(
        Mean_SNR_dB=(
            "SNR_dB",
            "mean"
        ),

        Std_SNR_dB=(
            "SNR_dB",
            "std"
        ),

        Total_Epochs=(
            "Trials",
            "sum"
        )
    )
    .reindex(task_order)
)

task_snr["Task_Label"] = [
    f"{task_labels[t]}\n"
    f"({task_epochs.get(t, 0):,} epochs)"
    for t in task_order
]


fig, ax = plt.subplots(
    figsize=(10, 6)
)

x = np.arange(
    len(task_order)
)

ax.errorbar(
    x,
    task_snr["Mean_SNR_dB"],
    yerr=task_snr["Std_SNR_dB"],
    marker="o",
    linestyle="none",
    capsize=4
)

ax.set_xticks(x)

ax.set_xticklabels(
    task_snr["Task_Label"]
)

ax.set_xlabel(
    "Task"
)

ax.set_ylabel(
    "Mean SNR (dB)"
)

ax.set_title(
    "Signal-to-Noise Ratio Across Tasks"
)

ax.grid(
    True,
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    PLOT_ROOT /
    "snr_by_task.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 7. ALL-RANK RECONSTRUCTION ERROR HEATMAP
# ============================================================

heatmap_data = (
    task_reconstruction_summary
    .pivot(
        index="Task",
        columns="Method",
        values="Mean_Reconstruction_Error_Percent"
    )
    .reindex(task_order)
    .reindex(columns=methods)
)

fig, ax = plt.subplots(
    figsize=(8, 6)
)

im = ax.imshow(
    heatmap_data.values,
    aspect="auto"
)


ax.set_xticks(
    np.arange(len(methods))
)

ax.set_xticklabels(
    methods
)

ax.set_yticks(
    np.arange(len(task_order))
)

ax.set_yticklabels([
    task_labels[t]
    for t in task_order
])

for i in range(
    heatmap_data.shape[0]
):

    for j in range(
        heatmap_data.shape[1]
    ):

        value = heatmap_data.iloc[i, j]

        if not np.isnan(value):

            ax.text(
                j,
                i,
                f"{value:.2f}%",
                ha="center",
                va="center"
            )


ax.set_xlabel(
    "Decomposition Method"
)

ax.set_ylabel(
    "Task"
)

ax.set_title(
    "Mean Reconstruction Error by Task and Decomposition\n"
    "(Averaged Across All Ranks)"
)


fig.colorbar(
    im,
    ax=ax,
    label="Reconstruction Error (%)"
)

plt.tight_layout()

plt.savefig(
    PLOT_ROOT /
    "reconstruction_error_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 8. RANK-SPECIFIC RECONSTRUCTION ERROR VS COMPRESSION
# ============================================================

task_rank_error_compression = (
    df_decomp
    .groupby(
        ["Task", "Method", "Rank"],
        dropna=False
    )
    .agg(
        Mean_Reconstruction_Error_Percent=(
            "Reconstruction_Error_Percent",
            "mean"
        ),

        Mean_Compression_Ratio=(
            "Compression_Ratio",
            "mean"
        )
    )
    .reset_index()
)


for method in methods:

    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    for task in task_order:

        subset = task_rank_error_compression[
            (
                task_rank_error_compression["Method"]
                == method
            )
            &
            (
                task_rank_error_compression["Task"]
                == task
            )
        ].sort_values(
            "Mean_Compression_Ratio"
        )


        if subset.empty:
            continue

        # Label each point with its rank
        for _, row in subset.iterrows():

            ax.annotate(
                f"r={row['Rank']}",
                (
                    row["Mean_Compression_Ratio"] * 100,
                    row["Mean_Reconstruction_Error_Percent"]
                ),
                xytext=(0, 8),
                textcoords="offset points",
                ha="center",
                fontsize=9
            )

        ax.plot(
            subset["Mean_Compression_Ratio"] * 100,
            subset[
                "Mean_Reconstruction_Error_Percent"
            ],
            marker="o",
            linewidth=2,
            label=task_labels[task]
        )


    ax.set_xlabel(
        "Mean Compression Ratio (%)"
    )

    ax.set_ylabel(
        "Mean Reconstruction Error (%)"
    )

    ax.set_title(
        f"{method} Reconstruction Error vs Compression "
        "Across Tasks"
    )

    ax.grid(
        True,
        alpha=0.3
    )

    ax.legend()

    plt.tight_layout()

    plt.savefig(
        PLOT_ROOT /
        f"reconstruction_error_vs_compression_by_task_{method}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# 9. RANK-SPECIFIC RECONSTRUCTION ERROR VS PARAMETERS
# ============================================================

task_rank_error_parameters = (
    df_decomp
    .groupby(
        ["Task", "Method", "Rank"],
        dropna=False
    )
    .agg(
        Mean_Reconstruction_Error_Percent=(
            "Reconstruction_Error_Percent",
            "mean"
        ),

        Mean_Compressed_Parameters=(
            "Compressed_Parameters",
            "mean"
        )
    )
    .reset_index()
)


for method in methods:

    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    for task in task_order:

        subset = task_rank_error_parameters[
            (
                task_rank_error_parameters["Method"]
                == method
            )
            &
            (
                task_rank_error_parameters["Task"]
                == task
            )
        ].sort_values(
            "Mean_Compressed_Parameters"
        )


        if subset.empty:
            continue


        ax.plot(
            subset["Mean_Compressed_Parameters"],
            subset[
                "Mean_Reconstruction_Error_Percent"
            ],
            marker="o",
            linewidth=2,
            label=task_labels[task]
        )


    ax.set_xscale(
        "log"
    )

    ax.set_xlabel(
        "Mean Number of Compressed Parameters"
    )

    ax.set_ylabel(
        "Mean Reconstruction Error (%)"
    )

    ax.set_title(
        f"{method} Reconstruction Error vs Model Size "
        "Across Tasks"
    )

    ax.grid(
        True,
        alpha=0.3
    )

    ax.legend()

    plt.tight_layout()

    plt.savefig(
        PLOT_ROOT /
        f"reconstruction_error_vs_parameters_by_task_{method}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# 10. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PLOTTING COMPLETE")
print("=" * 70)

print("\nExcel file:")
print(
    excel_path.resolve()
)

print("\nPlot directory:")
print(
    PLOT_ROOT.resolve()
)

print("\nDecomposition records:")
print(
    len(df_decomp)
)

print("\nSignal metric records:")
print(
    len(df_signal)
)

print("\nTask epoch totals:")

for task in task_order:

    print(
        f"  {task_labels[task]}: "
        f"{task_epochs.get(task, 0):,}"
    )

# ============================================================
# 11. COMPARISON OF ALL DECOMPOSITIONS:
#     RECONSTRUCTION ERROR VS COMPRESSED PARAMETERS
# ============================================================

comparison_data = (
    df_decomp
    .groupby(
        ["Method", "Rank"],
        dropna=False
    )
    .agg(
        Mean_Reconstruction_Error_Percent=(
            "Reconstruction_Error_Percent",
            "mean"
        ),

        Std_Reconstruction_Error_Percent=(
            "Reconstruction_Error_Percent",
            "std"
        ),

        Mean_Compressed_Parameters=(
            "Compressed_Parameters",
            "mean"
        )
    )
    .reset_index()
)


fig, ax = plt.subplots(
    figsize=(10, 7)
)


for method in methods:

    subset = comparison_data[
        comparison_data["Method"] == method
    ].sort_values(
        "Mean_Compressed_Parameters"
    )

    if subset.empty:
        continue

    ax.errorbar(
        subset["Mean_Compressed_Parameters"],
        subset["Mean_Reconstruction_Error_Percent"],
        yerr=subset["Std_Reconstruction_Error_Percent"],
        marker="o",
        linewidth=2,
        capsize=4,
        label=method
    )


ax.set_xscale("log")

ax.set_xlabel(
    "Mean Number of Compressed Parameters"
)

ax.set_ylabel(
    "Mean Reconstruction Error (%)"
)

ax.set_title(
    "Reconstruction Error vs Model Size"
)

ax.grid(
    True,
    which="both",
    alpha=0.3
)

ax.legend(
    title="Decomposition"
)

plt.tight_layout()

plt.savefig(
    PLOT_ROOT /
    "reconstruction_error_vs_parameters_all_methods.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ============================================================
# 12. RECONSTRUCTION ERROR VS MODEL SIZE
#     ALL METHODS, SEPARATED BY TASK
# ============================================================

task_comparison_data = (
    df_decomp
    .groupby(
        ["Task", "Method", "Rank"],
        dropna=False
    )
    .agg(
        Mean_Reconstruction_Error_Percent=(
            "Reconstruction_Error_Percent",
            "mean"
        ),

        Std_Reconstruction_Error_Percent=(
            "Reconstruction_Error_Percent",
            "std"
        ),

        Mean_Compressed_Parameters=(
            "Compressed_Parameters",
            "mean"
        )
    )
    .reset_index()
)


for task in task_order:

    fig, ax = plt.subplots(
        figsize=(10, 7)
    )

    for method in methods:

        subset = task_comparison_data[
            (
                task_comparison_data["Task"] == task
            )
            &
            (
                task_comparison_data["Method"] == method
            )
        ].sort_values(
            "Mean_Compressed_Parameters"
        )

        if subset.empty:
            continue

        ax.errorbar(
            subset["Mean_Compressed_Parameters"],
            subset["Mean_Reconstruction_Error_Percent"],
            yerr=subset["Std_Reconstruction_Error_Percent"],
            marker="o",
            linewidth=2,
            capsize=4,
            label=method
        )

    ax.set_xscale("log")

    ax.set_xlabel(
        "Mean Number of Compressed Parameters"
    )

    ax.set_ylabel(
        "Mean Reconstruction Error (%)"
    )

    ax.set_title(
        f"{task_labels[task]}: "
        "Reconstruction Error vs Model Size"
    )

    ax.grid(
        True,
        which="both",
        alpha=0.3
    )

    ax.legend(
        title="Decomposition"
    )

    plt.tight_layout()

    plt.savefig(
        PLOT_ROOT /
        f"reconstruction_error_vs_parameters_all_methods_{task}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


print("\n" + "=" * 70)
print("EXPERIMENT COMPLETE")
print("=" * 70)

print(f"Excel file:")
print(excel_path.resolve())

print("\nPlots:")
print(PLOT_ROOT.resolve())

print("\nRows in decomposition results:", len(df_decomp))
print("Signal metric records:", len(df_signal))


PLOTTING COMPLETE

Excel file:
/home/master/MasterThesis/OPM-MEG MPS/results/decomposition/decomposition_experiment4/tensor_decomposition_results.xlsx

Plot directory:
/home/master/MasterThesis/OPM-MEG MPS/results/decomposition/decomposition_experiment4/trial_plots

Decomposition records:
800

Signal metric records:
32

Task epoch totals:
  Auditory: 1,600
  Somatosensory: 1,629
  Motor: 850
  Rest: 1,774

EXPERIMENT COMPLETE
Excel file:
/home/master/MasterThesis/OPM-MEG MPS/results/decomposition/decomposition_experiment4/tensor_decomposition_results.xlsx

Plots:
/home/master/MasterThesis/OPM-MEG MPS/results/decomposition/decomposition_experiment4/trial_plots

Rows in decomposition results: 800
Signal metric records: 32
